# Lesson 5 — Regularization, and the "don't erase JNDI" prior trick

Lesson 5 — Regularization, and the "don't erase JNDI" prior trick
=================================================================

GURU NOTE
---------
Two problems appear the moment you train on real, limited data:

  1. OVERFITTING. With few examples, the model can grow huge weights to memorize
     quirks of THIS data that won't generalize. Fix: "regularization" — a penalty
     on big weights that keeps the model humble.

  2. RARE-BUT-CRITICAL SIGNALS GET ERASED. Standard regularization pulls weights
     toward ZERO. But some features (like `jndi` / Log4Shell) are rare in any
     corpus yet must stay damning. Pulling their weight to zero is exactly wrong.

Your repo solves #2 with a beautiful trick (train.py lines 33-34): regularize
toward a hand-written PRIOR (the shipped basic weights) instead of toward zero.
"Trust the expert's JNDI weight unless the data strongly argues otherwise."

This lesson demonstrates both, with a JNDI-like rare feature so you SEE the trick
save it.

Run me:  python lessons/lesson_05_regularization_and_priors.py

## Setup

These notebooks *import* the real code from `lessons/*.py` — one source of truth, no forked copies. This cell finds that folder.

In [1]:
import sys, pathlib
for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (base / 'lessons' / '_util.py').exists():
        sys.path.insert(0, str(base / 'lessons'))
        print('lessons on path:', base / 'lessons'); break
else:
    raise RuntimeError('Open this notebook inside the xorguard-ml-course folder.')

lessons on path: /Users/anil/Documents/developer/vscodeworkspaces/xordev/xorguard-ml-course/lessons


## The key code (shown live from the source file)

Read it, then copy any function into a new cell and tweak it — the originals live in `lessons/lesson_05_regularization_and_priors.py`.

In [2]:
import inspect
from lesson_05_regularization_and_priors import fit
for fn in (fit,):
    print(inspect.getsource(fn)); print('-' * 72)

def fit(X, y, l2: float = 0.0, prior=None, init=None, steps: int = 800, lr: float = 0.4):
    """Logistic regression with L2 regularization toward a PRIOR.

    This is train.fit_logistic's exact structure (minus class weights, Lesson 6):
        pen = l2 * (w - prior)      # pull toward prior, not necessarily zero
        w  -= step_lr * (grad + pen)
    prior=None  -> pull toward zero (standard L2).
    prior=vec   -> pull toward that vector (the repo's warm-start trick).
    """
    n, d = X.shape
    w = np.zeros(d) if init is None else np.array(init, float)
    prior = np.zeros(d) if prior is None else np.array(prior, float)
    for step_i in range(steps):
        p = sigmoid(X @ w)
        grad = X.T @ (p - y) / n
        pen = l2 * (w - prior)
        step_lr = lr / (1.0 + step_i / 250.0)
        w -= step_lr * (grad + pen)
    return w

------------------------------------------------------------------------


## Run the full narrated lesson

This executes the same walkthrough as `python lessons/lesson_05_regularization_and_priors.py`, printing each step with its result.

In [3]:
from lesson_05_regularization_and_priors import main
main()


═════════════════════════════════════════════
Lesson 5 — Regularization and the prior trick
═════════════════════════════════════════════


    dataset: 200 benign, 206 attacks (of which only 6 are JNDI-style rare).

▸ Part A — Overfitting and how L2 tames it.
    With NO regularization the model is free to grow large weights.
    ||w|| with l2=0.0 : 7.53  (bigger, more eager
    ||w|| with l2=0.5 : 0.35  to memorize)
    Regularization = deliberately keep weights small so the model
    generalizes instead of memorizing. (We'll MEASURE generalization
    properly with a train/val split in Lesson 7.)

▸ Part B — the problem: standard L2 (toward ZERO) erases the rare JNDI tell.
    Learned weights, L2 toward zero:
              sqli        |█████████████████████████████████ +0.287
           special        |██████████████████████            +0.193
              jndi        |█                                 +0.015
              bias ███████|                                  -0.059
    -> jndi weight = +0.015. Only 6 examples push it
       up, but the penalty pulls it toward 0 on every step. It's weak.

▸ Part C — the fix: r

## Exercises

Try these in new cells below. Worked answers are in `solutions/lesson_05_solution.py`.

```
1. Increase the number of JNDI examples from 6 to 100 and retrain the
   L2-to-zero model. Does jndi's weight recover? Explain why more data reduces
   the need for a prior.

2. Set the prior's jndi entry to a WRONG value (say -3.0) and retrain with lots
   of JNDI attacks in the data. Does the data eventually overrule the bad prior?
   This shows the prior is a bias, not a straitjacket.

3. Sweep l2 in {0.0, 0.1, 1.0, 10.0} with prior=None and print ||w|| for each.
   Describe the trade-off between fitting the data and staying humble.
```

In [4]:
# your work here
